# Notebook 2: Dataförberedelse

---

Ingen modell kan lära sig av smutsig data. I det här steget **rensar och transformerar** vi
datasetet så att det är redo för maskininlärning.

Stegen är:
1. Hantera outliers (inklusive ogiltiga 0-värden)
2. Koda om kategoriska variabler till siffror
3. Skala numeriska variabler
4. Dela upp i tränings- och testdata

## Ladda data

Vi läser alltid från råfilen – dataförberedelsen ska vara reproducerbar från grunden.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid', palette='muted')

df = pd.read_csv('Data/raw_heart_disease.csv')
print(f'Laddad: {df.shape[0]} rader, {df.shape[1]} kolumner')
df.head()

## Saknade värden

I Notebook 1 såg vi att det inte finns några NaN-värden.
Vi bekräftar det och dokumenterar det formellt.

In [ ]:
print('Saknade värden per kolumn:')
print(df.isnull().sum().to_string())
print(f'\nTotalt saknade värden: {df.isnull().sum().sum()}')

> **Bekräftat:** Inga NaN-värden. Men vi har ett annat datakvalitetsproblem som behöver
> hanteras – se nästa sektion.

## Outliers

Outliers är extremvärden som kan snedvrida modellerna. Vi hanterar två typer:

**1. Ogiltiga 0-värden** i `Cholesterol` och `RestingBP`
   - Kolesterol = 0 är medicinskt omöjligt → troligen saknat värde som kodats som 0
   - Vi ersätter dem med **medianen** av giltiga värden (mer robust än medelvärdet)

**2. Statistiska outliers** med IQR-metoden
   - Värden utanför `[Q1 - 1.5·IQR, Q3 + 1.5·IQR]` cappas till dessa gränser
   - Vi capper (begränsar) istället för att ta bort – vi vill inte tappa patienter

In [ ]:
# Steg 1: Ersätt ogiltiga 0-värden med medianen av giltiga värden
df_clean = df.copy()  # Aldrig modifiera originaldata direkt

for kol in ['Cholesterol', 'RestingBP']:
    n_nollor = (df_clean[kol] == 0).sum()
    if n_nollor > 0:
        # Beräkna median endast på giltiga (> 0) värden
        median_giltig = df_clean[df_clean[kol] > 0][kol].median()
        df_clean.loc[df_clean[kol] == 0, kol] = median_giltig
        print(f'{kol}: ersatte {n_nollor} nollor med medianen {median_giltig:.1f}')

In [ ]:
# Steg 2: IQR-cappning för numeriska variabler
# Vi visualiserar före/efter för att se effekten

def cap_outliers_iqr(series):
    """Cappa värden utanför 1.5 * IQR till gränsvärdena."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_capped = ((series < lower) | (series > upper)).sum()
    print(f'  {series.name}: {n_capped} värden cappade  (gränser: [{lower:.1f}, {upper:.1f}])')
    return series.clip(lower, upper)

numeriska_kols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
print('IQR-cappning:')
for kol in numeriska_kols:
    df_clean[kol] = cap_outliers_iqr(df_clean[kol])

In [ ]:
# Boxplots före vs efter för att visualisera förändringen
fig, axes = plt.subplots(2, len(numeriska_kols), figsize=(16, 8))
for i, kol in enumerate(numeriska_kols):
    axes[0][i].boxplot(df[kol], patch_artist=True,
        boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[0][i].set_title(f'{kol}\n(före)', fontsize=10)
    axes[1][i].boxplot(df_clean[kol], patch_artist=True,
        boxprops=dict(facecolor='tomato', alpha=0.7))
    axes[1][i].set_title(f'{kol}\n(efter)', fontsize=10)
plt.suptitle('Outlier-hantering: Före (blå) vs Efter (röd)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/02_outliers_fore_efter.png', dpi=150)
plt.show()

> **Fynd:** Cappningen reducerar extremvärden utan att ta bort patienter.
> `Cholesterol` och `RestingBP` förbättrades mest eftersom de hade 0-värden.

## Kodning av kategoriska variabler

Maskininlärningsmodeller räknar med siffror – de förstår inte text som `"M"` eller `"ASY"`.
Vi konverterar kategoriska variabler till binära (0/1) kolumner med **one-hot encoding**.

Exempel: `Sex` {M, F} → en kolumn `Sex_M` (1 om M, 0 om F)

`drop_first=True` tar bort en kolumn per variabel för att undvika **multikollinearitet**
(om `Sex_M = 0` vet vi automatiskt att det är F – vi behöver inte en extra kolumn för det).

In [ ]:
kategoriska_kols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

# pd.get_dummies skapar binära kolumner för varje unik kategori
df_encoded = pd.get_dummies(df_clean, columns=kategoriska_kols, drop_first=True, dtype=int)

print(f'Kolumner före encoding: {df_clean.shape[1]}')
print(f'Kolumner efter encoding: {df_encoded.shape[1]}')
print(f'\nNya kolumner: {df_encoded.columns.tolist()}')

## Skalning

KNN och Logistisk Regression är känsliga för skillnader i skala mellan variabler.
En variabel som mäts i hundratals (t.ex. `Cholesterol ≈ 200`) kan dominera
en som mäts i ental (t.ex. `Oldpeak ≈ 1`) – även om den inte är viktigare.

**StandardScaler** skalerar varje variabel till medelvärde 0 och standardavvikelse 1:
`z = (x − medelvärde) / standardavvikelse`

**Viktigt:** Vi fittar skalaren ENBART på träningsdata och applicerar samma
transformation på testdata. Annars läcker information från testdata in i modellen.

## Train/Test-split

Vi delar upp datan i:
- **Träningsdata (80%)** – modellen lär sig på dessa patienter
- **Testdata (20%)** – vi utvärderar modellen på patienter den aldrig sett

`stratify=y` säkerställer att klassbalansen (45%/55%) bevaras i båda delar.

In [ ]:
# Separera features (X) och målvariabel (y)
X = df_encoded.drop('HeartDisease', axis=1)
y = df_encoded['HeartDisease']

# Train/Test-split med stratifiering
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% till test
    random_state=42,      # Fixa slumpen så alla får samma split
    stratify=y            # Bevara klassbalansen
)

print(f'Träningsdata: {X_train.shape[0]} patienter')
print(f'Testdata:     {X_test.shape[0]} patienter')
print(f'\nKlassbalans i träningsdata:')
print(y_train.value_counts(normalize=True).round(3).to_string())
print(f'\nKlassbalans i testdata:')
print(y_test.value_counts(normalize=True).round(3).to_string())

In [ ]:
# Skala features – fit BARA på träningsdata, transform på båda
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)  # Lär sig medelvärde/std från träningsdata
X_test_scaled  = scaler.transform(X_test)        # Applicerar SAMMA transformation på testdata

print('Skalning klar!')
print(f'Medelvärde per feature (ska vara ~0): {X_train_scaled.mean(axis=0).round(2)}')

In [ ]:
# Spara processad data till disk – notebook 03 och 04 läser härifrån
feature_names = X.columns.tolist()

pd.DataFrame(X_train_scaled, columns=feature_names).to_csv(
    'Data/processed/X_train_scaled.csv', index=False)
pd.DataFrame(X_test_scaled, columns=feature_names).to_csv(
    'Data/processed/X_test_scaled.csv', index=False)
y_train.reset_index(drop=True).to_csv('Data/processed/y_train.csv', index=False)
y_test.reset_index(drop=True).to_csv('Data/processed/y_test.csv', index=False)

# Spara skalaren – kan behövas för att transformera ny data i framtiden
joblib.dump(scaler, 'outputs/models/scaler.pkl')

print('Sparade filer:')
print('  Data/processed/X_train_scaled.csv')
print('  Data/processed/X_test_scaled.csv')
print('  Data/processed/y_train.csv')
print('  Data/processed/y_test.csv')
print('  outputs/models/scaler.pkl')

## Sammanfattning Dataförberedelse

### Vad gjorde vi och varför?

| Steg | Åtgärd | Anledning |
|---|---|---|
| Ogiltiga 0-värden | Ersattes med medianen | Kolesterol/blodtryck = 0 är medicinskt omöjligt |
| IQR-cappning | Extremvärden begränsades | Minskar påverkan utan att tappa patienter |
| One-hot encoding | Kategorier → binära kolumner | Modeller kräver siffror |
| StandardScaler | Skalat till mean=0, std=1 | KNN och LR är skalänsliga |
| Train/Test-split | 80% / 20%, stratifierat | Rättvis utvärdering på osedd data |

---
**Nästa steg → Notebook 3: Modellering**
Vi tränar Logistisk Regression, KNN, Beslutsträd och Random Forest,
och testar sedan samma modeller kombinerade med PCA och UMAP.